# AI Fundamentals Exam

Total: 100 points. Time: 30 minutes. 10 multiple choice questions (2 points each) and 4 coding questions (20 points each).

**How to work**
- Run the setup cell and the grading engine cell first.
- For a multiple choice question, set the answer variable to `"A"`, `"B"`, `"C"` or `"D"`, run the cell, then run the collapsed grading cell directly below it.
- For a coding question, write your solution in the code cell, run it, then run the collapsed grading cell below it. Coding questions give partial credit per passed test.
- You may re-run a question; the last run counts and the number of attempts is recorded.
- Do not edit the collapsed grading cells.
- Run the last cell to see your final score.

**Data used in the coding questions**
- `titanic`: Titanic passenger records (891 rows), loaded from a public URL, so an internet connection is required.
- `tips`: restaurant tips data from `plotly.express`.

## Setup

In [81]:
student_name = "محمد فراس أمين"  # write your full name between the quotes

In [82]:
# pip install pandas numpy requests plotly
import json
import os
import tempfile

import numpy as np
import pandas as pd
import requests
import plotly.express as px

titanic = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")
tips = px.data.tips()

print("titanic:", titanic.shape)
print(list(titanic.columns))
print("tips:", tips.shape)
print(list(tips.columns))

titanic: (891, 15)
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
tips: (244, 7)
['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']


In [83]:
#@title Grading engine (do not edit)
import hashlib

_SALT = "ai-fund-exam-v1"
_POINTS = {"Q1": 2, "Q2": 2, "Q3": 2, "Q4": 2, "Q5": 2, "Q6": 2, "Q7": 2, "Q8": 2, "Q9": 2, "Q10": 2, "C1": 20, "C2": 20, "C3": 20, "C4": 20}
_TOPIC = {"Q1": "Python Fundamentals", "Q2": "Python Fundamentals", "Q3": "Files, JSON and OOP", "Q4": "Files, JSON and OOP", "Q5": "APIs", "Q6": "APIs", "Q7": "Hugging Face", "Q8": "Pandas", "Q9": "Data Cleaning", "Q10": "Plotly", "C1": "APIs", "C2": "Pandas", "C3": "Data Cleaning", "C4": "Plotly"}
_KEY = {"Q1": "35769541745fbd119acb27dc4c0a1c1657c6bee17ffcf9efff3d30dbc63607b7", "Q2": "fb6ded9ecf5a75c951966b9e96d586f11fd13027203ddec1c1aae660c528355d", "Q3": "7d54ee3dea77de0b9a5b866832141ff3594464de1bb4a00fab96ba48ee9e77e7", "Q4": "2a3f471fe3eab5f5675ecc41eb950f9d8df86abb254174ba6062ea42ff28f1bf", "Q5": "e4bf4c0310424ba69ed68cb3a7ecc4a4fc9d61c9344c0cb1905dd6445710fc6a", "Q6": "5a2be2f17eda4820e5c9ca17fc983f30613db29397eac863995760e6a591157a", "Q7": "88ad1e93e4fe334afe054985e42fd267ede38140651b204a58f8449ba7b1defc", "Q8": "4f822591f0800a2ba96c4fc5f0c86f7b6f6b5de42b73532c5d25ee7ff1fcb5b6", "Q9": "64a0b3efaf0bd1044b1fdfd09b80b74f51ffa1472685781528e4f15c2bdb8afb", "Q10": "7a32709c53d819d2b8de42055bc592bc31315cc65e36f73dd494a5872ce432c2"}
_RESULTS = {}


def _digest(qid, answer):
    return hashlib.sha256(f"{_SALT}|{qid}|{str(answer).strip().upper()}".encode("utf-8")).hexdigest()


def _store(qid, earned, status):
    attempts = _RESULTS.get(qid, {}).get("attempts", 0) + 1
    _RESULTS[qid] = {"earned": round(earned, 2), "status": status, "attempts": attempts}


def grade_mcq(qid, answer):
    pts = _POINTS[qid]
    choice = str(answer).strip().upper()
    if choice not in ("A", "B", "C", "D"):
        _store(qid, 0, "no valid answer")
        print(f"{qid} | No valid answer | 0/{pts}")
        print("Set the answer variable to 'A', 'B', 'C' or 'D' and run the cell again.")
    elif _digest(qid, choice) == _KEY[qid]:
        _store(qid, pts, "correct")
        print(f"{qid} | Correct | {pts}/{pts}")
    else:
        _store(qid, 0, "incorrect")
        print(f"{qid} | Incorrect | 0/{pts}")


def grade_code(qid, tests):
    pts = _POINTS[qid]
    passed, lines = 0, []
    for desc, fn in tests:
        try:
            ok = bool(fn())
            passed += ok
            lines.append(("pass" if ok else "FAIL", desc, ""))
        except Exception as e:
            lines.append(("ERROR", desc, f"{type(e).__name__}: {e}"))
    earned = pts * passed / len(tests)
    _store(qid, earned, f"{passed}/{len(tests)} tests")
    print(f"{qid} | {passed}/{len(tests)} tests passed | {earned:.1f}/{pts}")
    for status, desc, err in lines:
        print(f"  [{status}] {desc}" + (f" -> {err}" if err else ""))


def final_report():
    name = student_name.strip() if student_name.strip() else "(name not set)"
    print(f"Student: {name}")
    print("-" * 62)
    print(f"{'Question':<10}{'Topic':<26}{'Score':>10}{'Attempts':>12}")
    print("-" * 62)
    for qid in _POINTS:
        r = _RESULTS.get(qid)
        score = f"{r['earned']:.1f}/{_POINTS[qid]}" if r else f"0/{_POINTS[qid]}"
        att = r["attempts"] if r else "not run"
        print(f"{qid:<10}{_TOPIC[qid]:<26}{score:>10}{str(att):>12}")
    print("-" * 62)
    total = sum(r["earned"] for r in _RESULTS.values())
    print(f"TOTAL: {total:.1f} / {sum(_POINTS.values())}")
    missing = [q for q in _POINTS if q not in _RESULTS]
    if missing:
        print("Not graded yet: " + ", ".join(missing))


## Section 1. Python Fundamentals

### Q1 (2 points)

What does this code print?

```python
ai_terms = {"AI": "Artificial Intelligence", "ML": "Machine Learning"}
print(ai_terms.get("DL", "Unknown"), len(ai_terms))
```

- **A.** Unknown 2
- **B.** None 2
- **C.** KeyError
- **D.** Unknown 3

In [84]:
answer_q1 = "A"  # "A", "B", "C" or "D"

In [85]:
#@title Grade Q1 (do not edit)
grade_mcq("Q1", answer_q1)

Q1 | Correct | 2/2


### Q2 (2 points)

What does this code print?

```python
print([n ** 2 for n in range(6) if n % 2 == 0])
```

- **A.** [0, 1, 4, 9, 16, 25]
- **B.** [1, 9, 25]
- **C.** [4, 16]
- **D.** [0, 4, 16]

In [86]:
answer_q2 = "D"  # "A", "B", "C" or "D"

In [87]:
#@title Grade Q2 (do not edit)
grade_mcq("Q2", answer_q2)

Q2 | Correct | 2/2


## Section 2. File Handling, JSON and OOP

### Q3 (2 points)

The file `notes.txt` contains three lines. You then run the code below. What does the file contain afterward?

```python
with open("notes.txt", "w") as f:
    f.write("Done\n")
```

- **A.** The three original lines followed by Done
- **B.** Only Done
- **C.** Done followed by the three original lines
- **D.** The file is unchanged, because mode "w" is read-only

In [88]:
answer_q3 = "B"  # "A", "B", "C" or "D"

In [89]:
#@title Grade Q3 (do not edit)
grade_mcq("Q3", answer_q3)

Q3 | Correct | 2/2


### Q4 (2 points)

What does this code print?

```python
class Message:
    def __init__(self, sender, text):
        self.sender = sender
        self.text = text

    def __str__(self):
        return f"{self.sender}: {self.text.upper()}"


msgs = [Message("User", "hi"), Message("AI", "hello")]
print(msgs[1])
print(len(msgs))
```

- **A.** AI: hello, then 2
- **B.** AI: HELLO, then 2
- **C.** `<__main__.Message object at 0x...>`, then 2
- **D.** AI: HELLO, then 1

In [90]:
answer_q4 = "B"  # "A", "B", "C" or "D"

In [91]:
#@title Grade Q4 (do not edit)
grade_mcq("Q4", answer_q4)

Q4 | Correct | 2/2


## Section 3. APIs

### Q5 (2 points)

`requests.get(url)` on a public Wikipedia endpoint returns status 403, but the same URL opens fine in a browser. What is the most likely cause and fix?

- **A.** The server rejects the default Python client identity; send a descriptive User-Agent header
- **B.** The API key is missing; append it to the URL
- **C.** The rate limit was exceeded; wait and retry
- **D.** The server crashed (a 5xx problem); retry later

In [92]:
answer_q5 = "A"  # "A", "B", "C" or "D"

In [93]:
#@title Grade Q5 (do not edit)
grade_mcq("Q5", answer_q5)

Q5 | Correct | 2/2


### Q6 (2 points)

Which is the safest way to handle an API key in a notebook you will share or push to GitHub?

- **A.** Hardcode it in a variable in the first cell
- **B.** Base64-encode it and paste the encoded string in the notebook
- **C.** Hardcode it, run the cell, then clear the cell output
- **D.** Store it in a `.env` file, load it with python-dotenv, and add `.env` to `.gitignore`

In [94]:
answer_q6 = "D"  # "A", "B", "C" or "D"

In [95]:
#@title Grade Q6 (do not edit)
grade_mcq("Q6", answer_q6)

Q6 | Correct | 2/2


### C1 (20 points)

Write `parse_gemini_response(response)` for a `requests` response object (it has `.status_code` and `.json()`).

- Status `200` and the JSON has a reply: return the reply text with surrounding whitespace removed. The text is at `["candidates"][0]["content"]["parts"][0]["text"]`.
- Status `200` but the JSON has no usable `candidates` (key missing or empty list): return `"Unexpected response format"`
- Status `401` or `403`: return `"Authentication failed"`
- Status `429`: return `"Rate limit exceeded"`
- Any other status of 400 or above: return `f"Request failed with status {status_code}"`

Do not call `.json()` on error responses; they may not contain JSON.

In [122]:
def parse_gemini_response(response):
    try:
        if response.status_code == 200:
            result = response.json()
            candidates = result["candidates"]
            if candidates:
                try:
                    return result["candidates"][0]["content"]["parts"][0]["text"].strip()
                except Exception as e:
                    return "Unexpected response format"

        if response.status_code == 401 or response.status_code == 403:
            return "Authentication failed"

        if response.status_code == 429:
            return "Rate limit exceeded"

        else:
            return f"Request failed with status {response.status_code}"
    except requests.exceptions.ConnectionError:
        return "Connection ERROR!!"
    except requests.exceptions.Timeout:
        return "TimeOUT ERROR!!"
    except Exception as e:
        return f"[ERROR]: {e}"

In [123]:
#@title Grade C1 (do not edit)
class _FakeResponse:
    def __init__(self, status_code, payload=None):
        self.status_code = status_code
        self._payload = payload

    def json(self):
        if self._payload is None:
            raise ValueError("No JSON body")
        return self._payload

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.exceptions.HTTPError(f"{self.status_code} Error", response=self)

_ok = {"candidates": [{"content": {"parts": [{"text": "  Machine learning learns patterns from data.  "}], "role": "model"}}]}

def _t1(): return parse_gemini_response(_FakeResponse(200, _ok)) == "Machine learning learns patterns from data."
def _t2(): return parse_gemini_response(_FakeResponse(200, {"candidates": []})) == "Unexpected response format"
def _t3(): return parse_gemini_response(_FakeResponse(200, {"error": {"message": "bad"}})) == "Unexpected response format"
def _t4(): return all(parse_gemini_response(_FakeResponse(c)) == "Authentication failed" for c in (401, 403))
def _t5(): return parse_gemini_response(_FakeResponse(429)) == "Rate limit exceeded"
def _t6(): return parse_gemini_response(_FakeResponse(500)) == "Request failed with status 500"

grade_code("C1", [
    ("200 returns the stripped reply text", _t1),
    ("200 with an empty candidates list", _t2),
    ("200 with no candidates key", _t3),
    ("401 and 403 return the authentication message", _t4),
    ("429 returns the rate limit message", _t5),
    ("other error codes include the status code", _t6),
])

C1 | 4/6 tests passed | 13.3/20
  [pass] 200 returns the stripped reply text
  [FAIL] 200 with an empty candidates list
  [FAIL] 200 with no candidates key
  [pass] 401 and 403 return the authentication message
  [pass] 429 returns the rate limit message
  [pass] other error codes include the status code


## Section 4. Hugging Face

### Q7 (2 points)

Which statement about running a model locally with `pipeline()` versus calling the serverless Inference API is correct?

- **A.** Local inference needs an internet connection for every prediction
- **B.** Local inference keeps your data on your machine, but needs disk space and compute for the model weights
- **C.** The Inference API has no rate limits, so a token is never needed
- **D.** The Inference API runs the model on your own GPU

In [98]:
answer_q7 = "B"  # "A", "B", "C" or "D"

In [99]:
#@title Grade Q7 (do not edit)
grade_mcq("Q7", answer_q7)

Q7 | Correct | 2/2


## Section 5. Pandas

### Q8 (2 points)

Why does `df[(df["a"] > 1) & (df["b"] < 5)]` use `&` with parentheses instead of Python's `and`?

- **A.** `and` is slower than `&` but gives the same result
- **B.** The parentheses are only a style convention
- **C.** `&` works element-wise on Boolean Series, while `and` needs one True/False value; and `&` binds tighter than comparisons, so each comparison needs parentheses
- **D.** `&` first converts both columns to integers

In [100]:
answer_q8 = "C"  # "A", "B", "C" or "D"

In [101]:
#@title Grade Q8 (do not edit)
grade_mcq("Q8", answer_q8)

Q8 | Correct | 2/2


### C2 (20 points)

Use the `titanic` DataFrame loaded in the setup cell. Write two functions. Neither may modify the DataFrame it receives.

`adult_first_class(df)` returns the rows where `pclass` is 1 **and** `age` is greater than 40.

`class_summary(df)` returns a DataFrame indexed by `pclass` with exactly these columns, in this order:
- `n_passengers`: number of passengers in the class
- `survival_rate`: mean of `survived`, rounded to 3 decimals
- `avg_fare`: mean of `fare`, rounded to 2 decimals

Sort the result by `survival_rate`, highest first.

In [115]:
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [116]:
def adult_first_class(df: pd.DataFrame):
    return df[(df["pclass"] == 1) & (df["age"] > 40)]


def class_summary(df: pd.DataFrame):
    new_df = (df.groupby("pclass").agg(
        n_passengers = ("survived", "size"),
        survival_rate = ("survived", "mean"),
        avg_fare = ("fare", "mean"),
    ).round({"survival_rate": 3, "avg_fare": 2})
    .sort_values("survival_rate", ascending=False)
    )
    return new_df

In [117]:
#@title Grade C2 (do not edit)
_ref_n = int(((titanic["pclass"] == 1) & (titanic["age"] > 40)).sum())
_ref_sum = (titanic.groupby("pclass")
            .agg(n_passengers=("survived", "size"), survival_rate=("survived", "mean"), avg_fare=("fare", "mean"))
            .round({"survival_rate": 3, "avg_fare": 2})
            .sort_values("survival_rate", ascending=False))

def _t1(): return len(adult_first_class(titanic.copy())) == _ref_n
def _t2():
    r = adult_first_class(titanic.copy())
    return len(r) > 0 and bool((r["pclass"] == 1).all()) and bool((r["age"] > 40).all())
def _t3():
    d = titanic.copy()
    a, b = adult_first_class(d), class_summary(d)
    return a is not None and b is not None and d.shape == titanic.shape and list(d.columns) == list(titanic.columns)
def _t4(): return list(class_summary(titanic.copy()).columns) == ["n_passengers", "survival_rate", "avg_fare"]
def _t5():
    r = class_summary(titanic.copy())
    try:
        pd.testing.assert_frame_equal(r.sort_index(), _ref_sum.sort_index(), check_dtype=False)
        return True
    except AssertionError:
        return False
def _t6(): return list(class_summary(titanic.copy()).index) == list(_ref_sum.index)

grade_code("C2", [
    ("adult_first_class returns the right number of rows", _t1),
    ("adult_first_class applies both conditions", _t2),
    ("input DataFrame is not modified", _t3),
    ("class_summary has the required columns", _t4),
    ("class_summary values are correct", _t5),
    ("class_summary is sorted by survival_rate, highest first", _t6),
])

C2 | 6/6 tests passed | 20.0/20
  [pass] adult_first_class returns the right number of rows
  [pass] adult_first_class applies both conditions
  [pass] input DataFrame is not modified
  [pass] class_summary has the required columns
  [pass] class_summary values are correct
  [pass] class_summary is sorted by survival_rate, highest first


## Section 6. Data Cleaning

### Q9 (2 points)

A numeric column has Q1 = 20 and Q3 = 60. Using the 1.5 x IQR rule, which value is flagged as an outlier?

- **A.** 100
- **B.** 110
- **C.** -30
- **D.** 125

In [104]:
q1 = 20
q3 = 60 
iqr = q3 - q1 
print(q1 - 1.5 * iqr)
print(1.5 * iqr + q3)

-40.0
120.0


In [105]:
answer_q9 = "D"  # "A", "B", "C" or "D"

In [106]:
#@title Grade Q9 (do not edit)
grade_mcq("Q9", answer_q9)

Q9 | Correct | 2/2


### C3 (20 points)

Write `clean_titanic(df)` that returns a cleaned copy of the Titanic DataFrame. The input must not be modified. Apply these steps **in this order**:

1. Remove exact duplicate rows
2. Fill missing `age` values with the median `age` of the passenger's `pclass` (computed after step 1)
3. Fill missing `embarked` values with the most frequent value of `embarked`
4. Drop the `deck` column
5. Rename `sibsp` to `siblings_spouses` and `parch` to `parents_children`

Return the result with a reset index (`drop=True`).

In [107]:
def clean_titanic(df: pd.DataFrame):
    new_df = df.copy()
    new_df.drop_duplicates(keep="first", inplace=True)
    new_df["age"] = new_df["age"].fillna(new_df.groupby("pclass")["age"].transform("median"))
    new_df["embarked"] = new_df["embarked"].fillna(new_df["embarked"].mode()[0])
    new_df.drop(columns=["deck"], inplace=True)
    new_df.rename(
        columns={
            "sibsp": "siblings_spouses",
            "parch": "parents_children",
        },
        inplace=True
    )
    new_df.head(10)
    return new_df.reset_index(drop=True)

In [108]:
clean_titanic(titanic)

,survived,pclass,sex,age,siblings_spouses,parents_children,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
779,0,3,female,39.0,0,5,29.1250,Q,Third,woman,False,Queenstown,no,False
780,1,1,female,19.0,0,0,30.0000,S,First,woman,False,Southampton,yes,True
781,0,3,female,24.0,1,2,23.4500,S,Third,woman,False,Southampton,no,False
782,1,1,male,26.0,0,0,30.0000,C,First,man,True,Cherbourg,yes,True


In [109]:
#@title Grade C3 (do not edit)
def _ref_clean(df):
    d = df.drop_duplicates().copy()
    d["age"] = d["age"].fillna(d.groupby("pclass")["age"].transform("median"))
    d["embarked"] = d["embarked"].fillna(d["embarked"].mode()[0])
    d = d.drop(columns=["deck"])
    d = d.rename(columns={"sibsp": "siblings_spouses", "parch": "parents_children"})
    return d.reset_index(drop=True)

_ref = _ref_clean(titanic)

def _t1():
    r = clean_titanic(titanic.copy())
    return len(r) == len(titanic.drop_duplicates()) == len(_ref)
def _t2():
    r = clean_titanic(titanic.copy())
    return int(r["age"].isna().sum()) == 0 and abs(float(r["age"].sum()) - float(_ref["age"].sum())) < 0.01
def _t3():
    r = clean_titanic(titanic.copy())
    return int(r["embarked"].isna().sum()) == 0 and r["embarked"].value_counts().to_dict() == _ref["embarked"].value_counts().to_dict()
def _t4():
    r = clean_titanic(titanic.copy())
    return "deck" not in r.columns and len(r.columns) == len(_ref.columns)
def _t5():
    r = clean_titanic(titanic.copy())
    return (set(r.columns) == set(_ref.columns)
            and "sibsp" not in r.columns and "parch" not in r.columns
            and int(r["siblings_spouses"].sum()) == int(_ref["siblings_spouses"].sum())
            and int(r["parents_children"].sum()) == int(_ref["parents_children"].sum()))
def _t6():
    r = clean_titanic(titanic.copy())
    return isinstance(r, pd.DataFrame) and list(r.index) == list(range(len(r)))
def _t7():
    d = titanic.copy()
    r = clean_titanic(d)
    return isinstance(r, pd.DataFrame) and d.shape == titanic.shape and "deck" in d.columns and int(d["age"].isna().sum()) == int(titanic["age"].isna().sum())

grade_code("C3", [
    ("duplicates removed", _t1),
    ("age filled with the median of each pclass", _t2),
    ("embarked filled with the most frequent value", _t3),
    ("deck dropped", _t4),
    ("sibsp and parch renamed, other columns kept", _t5),
    ("index is reset", _t6),
    ("input DataFrame is not modified", _t7),
])

C3 | 7/7 tests passed | 20.0/20
  [pass] duplicates removed
  [pass] age filled with the median of each pclass
  [pass] embarked filled with the most frequent value
  [pass] deck dropped
  [pass] sibsp and parch renamed, other columns kept
  [pass] index is reset
  [pass] input DataFrame is not modified


## Section 7. Plotly

### Q10 (2 points)

You need to compare the median and spread of order sales across three product categories side by side. Which chart fits best?

- **A.** Pie chart
- **B.** Treemap
- **C.** Box plot
- **D.** Line chart

In [119]:
answer_q10 = "C"  # "A", "B", "C" or "D"

In [120]:
#@title Grade Q10 (do not edit)
grade_mcq("Q10", answer_q10)

Q10 | Correct | 2/2


### C4 (20 points)

Use the `tips` DataFrame loaded in the setup cell (columns include `total_bill`, `tip`, `day`). Write `make_figures(df)` that returns a tuple `(bar_fig, hist_fig)` built with Plotly Express.

`bar_fig`: a bar chart of the **total** `total_bill` for each `day`, with `day` on the x-axis, the title `"Total Bill by Day"`, and a single trace (do not use the `color` argument).

`hist_fig`: a histogram of `tip` with 20 bins and the title `"Tip Distribution"`.

In [112]:
def make_figures(df: pd.DataFrame):
    total_bills = df.groupby("day", as_index=False)["total_bill"].sum()
    bar_fig = px.bar(
        data_frame=total_bills,
        x = "day",
        y = "total_bill",
        title = "Total Bill by Day"
    )
    hist_fig = px.histogram(
        data_frame=df,
        x="tip",
        nbins=20,
        title="Tip Distribution"
    )
    return (bar_fig, hist_fig)

In [113]:
#@title Grade C4 (do not edit)
import plotly.graph_objects as go

_ref_totals = tips.groupby("day")["total_bill"].sum().to_dict()

def _t1():
    b, h = make_figures(tips)
    return isinstance(b, go.Figure) and isinstance(h, go.Figure)
def _t2():
    b, h = make_figures(tips)
    return len(b.data) == 1 and b.data[0].type == "bar"
def _t3():
    b, h = make_figures(tips)
    got = dict(zip(list(b.data[0].x), list(b.data[0].y)))
    return set(got) == set(_ref_totals) and all(abs(got[k] - _ref_totals[k]) < 1e-6 for k in _ref_totals)
def _t4():
    b, h = make_figures(tips)
    return b.layout.title.text == "Total Bill by Day"
def _t5():
    b, h = make_figures(tips)
    return h.data[0].type == "histogram" and h.data[0].nbinsx == 20
def _t6():
    b, h = make_figures(tips)
    return list(h.data[0].x) == list(tips["tip"]) and h.layout.title.text == "Tip Distribution"

grade_code("C4", [
    ("both return values are Plotly figures", _t1),
    ("bar chart has a single bar trace", _t2),
    ("bar heights are the total bill per day", _t3),
    ("bar chart title", _t4),
    ("histogram trace with 20 bins", _t5),
    ("histogram uses tip and has the right title", _t6),
])

C4 | 6/6 tests passed | 20.0/20
  [pass] both return values are Plotly figures
  [pass] bar chart has a single bar trace
  [pass] bar heights are the total bill per day
  [pass] bar chart title
  [pass] histogram trace with 20 bins
  [pass] histogram uses tip and has the right title


## Final Score

In [121]:
#@title Final score (do not edit)
final_report()

Student: محمد فراس أمين
--------------------------------------------------------------
Question  Topic                          Score    Attempts
--------------------------------------------------------------
Q1        Python Fundamentals            2.0/2           1
Q2        Python Fundamentals            2.0/2           1
Q3        Files, JSON and OOP            2.0/2           1
Q4        Files, JSON and OOP            2.0/2           1
Q5        APIs                           2.0/2           1
Q6        APIs                           2.0/2           1
Q7        Hugging Face                   2.0/2           1
Q8        Pandas                         2.0/2           1
Q9        Data Cleaning                  2.0/2           1
Q10       Plotly                         2.0/2           2
C1        APIs                         16.7/20           1
C2        Pandas                       20.0/20           2
C3        Data Cleaning                20.0/20           1
C4        Plotly        